In [0]:
df1 = spark.read.table("demo.bronze.nyc_taxi")
display(df1)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Trim string-type columns in df1 to remove leading/trailing spaces
for field in df1.schema.fields:
    print(f"{field.name}: {field.dataType}")
    if isinstance(field.dataType, StringType):
        df1 = df1.withColumn(field.name, F.trim(F.col(field.name)))





In [0]:
# Insert records that are missing in the silver table 
df1.createOrReplaceTempView("temp_tbl")
spark.sql("""    
          Insert into demo.silver.nyc_taxi
          select A.* from temp_tbl A 
          left join demo.silver.nyc_taxi B 
          on A.VendorID = B.VendorID
          and A.tpep_pickup_datetime = B.pickup_datetime
          and A.tpep_dropoff_datetime = B.dropoff_datetime
          where B.VendorID is null
          """)
